In [ ]:
!pip install numpy datasets gensim scikit-learn transformers torch tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 22.3 MB/s eta 0:00:00


In [ ]:
import numpy as np
import re
import torch
from datasets import load_dataset
from gensim.models import Word2Vec
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
from transformers import BertTokenizer, BertModel
from tqdm import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

data = load_dataset("emotion", trust_remote_code=True)

train_texts = data['train']['text']
train_labels = data['train']['label']
test_texts = data['validation']['text']
test_labels = data['validation']['label']

def tokenize_text(text):
    return re.findall(r'\b\w+\b', text.lower())

train_sentences = [tokenize_text(text) for text in train_texts]
test_sentences = [tokenize_text(text) for text in test_texts]

w2v_model = Word2Vec(
    sentences=train_sentences,
    vector_size=100,
    window=5,
    min_count=5,
    workers=4,
    sg=1
)

def get_w2v_mean_embeddings(sentences, model, vector_size):
    embeddings = []
    for sentence in sentences:
        valid_words = [word for word in sentence if word in model.wv.key_to_index]
        if valid_words:
            mean_vec = np.mean(model.wv[valid_words], axis=0)
        else:
            mean_vec = np.zeros(vector_size)
        embeddings.append(mean_vec)
    return np.array(embeddings)

X_train_w2v = get_w2v_mean_embeddings(train_sentences, w2v_model, 100)
X_test_w2v = get_w2v_mean_embeddings(test_sentences, w2v_model, 100)

clf_w2v = Pipeline([
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(max_iter=2000, C=5, class_weight='balanced'))
])

clf_w2v.fit(X_train_w2v, train_labels)

preds_w2v = clf_w2v.predict(X_test_w2v)
f1_w2v_macro = f1_score(test_labels, preds_w2v, average='macro')
f1_w2v_weighted = f1_score(test_labels, preds_w2v, average='weighted')


tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertModel.from_pretrained('bert-base-uncased')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
bert_model.to(device)
bert_model.eval()

def get_bert_embeddings(texts, batch_size=64):
    cls_embeddings = []
    mean_embeddings = []

    for i in tqdm(range(0, len(texts), batch_size), desc="Обработка текстов BERT"):
        batch_texts = texts[i:i + batch_size]

        # Токенизируем батч
        inputs = tokenizer(batch_texts, padding=True, truncation=True, max_length=256, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = bert_model(**inputs)

        last_hidden = outputs.last_hidden_state

        batch_cls = last_hidden[:, 0, :].cpu().numpy()
        cls_embeddings.extend(batch_cls)

        mask = inputs['attention_mask'].unsqueeze(-1).expand(last_hidden.size()).float()

        weights = torch.softmax(last_hidden.masked_fill(mask == 0, -1e9), dim=1)

        emb = torch.sum(last_hidden * weights, dim=1)

        batch_weighted = emb.cpu().numpy()
        mean_embeddings.extend(batch_weighted)

    return np.array(cls_embeddings), np.array(mean_embeddings)

X_train_bert_cls, X_train_bert_mean = get_bert_embeddings(train_texts)

X_test_bert_cls, X_test_bert_mean = get_bert_embeddings(test_texts)

clf_bert_cls = Pipeline([
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(max_iter=2000, C=5, class_weight='balanced'))
])
clf_bert_cls.fit(X_train_bert_cls, train_labels)

preds_bert_cls = clf_bert_cls.predict(X_test_bert_cls)
f1_bert_cls_macro = f1_score(test_labels, preds_bert_cls, average='macro')
f1_bert_cls_weighted = f1_score(test_labels, preds_bert_cls, average='weighted')

clf_bert_mean = Pipeline([
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(max_iter=2000, C=5, class_weight='balanced'))
])
clf_bert_mean.fit(X_train_bert_mean, train_labels)

preds_bert_mean = clf_bert_mean.predict(X_test_bert_mean)
f1_bert_mean_macro = f1_score(test_labels, preds_bert_mean, average='macro')
f1_bert_mean_weighted = f1_score(test_labels, preds_bert_mean, average='weighted')

print("\nРЕЗУЛЬТАТЫ")
print(f"Word2Vec (macro):    {f1_w2v_macro:.4f}")
print(f"Word2Vec (weighted): {f1_w2v_weighted:.4f}\n")

print(f"BERT CLS (macro):    {f1_bert_cls_macro:.4f}")
print(f"BERT CLS (weighted): {f1_bert_cls_weighted:.4f}\n")

print(f"BERT mean (macro):   {f1_bert_mean_macro:.4f}")
print(f"BERT mean (weighted):{f1_bert_mean_weighted:.4f}")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'emotion' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'emotion' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all

README.md: 0.00B [00:00, ?B/s]

split/train-00000-of-00001.parquet:   0%|          | 0.00/1.03M [00:00<?, ?B/s]

split/validation-00000-of-00001.parquet:   0%|          | 0.00/127k [00:00<?, ?B/s]

split/test-00000-of-00001.parquet:   0%|          | 0.00/129k [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Обработка текстов BERT: 100%|██████████| 32/32 [00:07<00:00,  4.09it/s]



РЕЗУЛЬТАТЫ
Word2Vec (macro):    0.3185
Word2Vec (weighted): 0.3924

BERT CLS (macro):    0.4709
BERT CLS (weighted): 0.5402

BERT mean (macro):   0.5170
BERT mean (weighted):0.5909


In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.linear_model import SGDClassifier
from sklearn.neighbors import KNeighborsClassifier

classifiers = {
    "MultinomialNB": MultinomialNB(),

    "LinearSVM": Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LinearSVC(
            C=1.0,
            class_weight='balanced',
            max_iter=5000
        ))
    ]),

    "SGDClassifier": Pipeline([
        ('scaler', StandardScaler()),
        ('clf', SGDClassifier(
            max_iter=5000,
            class_weight='balanced',
            random_state=42
        ))
    ]),

    "kNN": Pipeline([
        ('scaler', StandardScaler()),
        ('clf', KNeighborsClassifier(
            n_neighbors=5
        ))
    ])
}

def evaluate_classifiers(X_train, y_train, X_test, y_test, feature_name):
    print(f"Признаки: {feature_name}")

    X_train_nb = X_train.copy()
    X_test_nb = X_test.copy()

    min_value = min(X_train.min(), X_test.min())
    if min_value < 0:
        X_train_nb = X_train_nb - min_value
        X_test_nb = X_test_nb - min_value

    for clf_name, clf in classifiers.items():
        print(f"\nОбучение {clf_name}...")

        if clf_name == "MultinomialNB":
            clf.fit(X_train_nb, y_train)
            preds = clf.predict(X_test_nb)
        else:
            clf.fit(X_train, y_train)
            preds = clf.predict(X_test)

        macro = f1_score(y_test, preds, average='macro')
        weighted = f1_score(y_test, preds, average='weighted')

        print(f"{clf_name:20s} | macro: {macro:.4f} | weighted: {weighted:.4f}")

# Word2Vec embeddings
evaluate_classifiers(
    X_train_w2v,
    train_labels,
    X_test_w2v,
    test_labels,
    "Word2Vec mean embeddings"
)

# BERT CLS embeddings
evaluate_classifiers(
    X_train_bert_cls,
    train_labels,
    X_test_bert_cls,
    test_labels,
    "BERT CLS embeddings"
)

# BERT weighted mean embeddings
evaluate_classifiers(
    X_train_bert_mean,
    train_labels,
    X_test_bert_mean,
    test_labels,
    "BERT weighted mean embeddings"
)

Признаки: Word2Vec mean embeddings

Обучение MultinomialNB...
MultinomialNB        | macro: 0.0868 | weighted: 0.1833

Обучение LinearSVM...
LinearSVM            | macro: 0.3425 | weighted: 0.4396

Обучение SGDClassifier...
SGDClassifier        | macro: 0.2680 | weighted: 0.3488

Обучение kNN...
kNN                  | macro: 0.1950 | weighted: 0.3022
Признаки: BERT CLS embeddings

Обучение MultinomialNB...
MultinomialNB        | macro: 0.1596 | weighted: 0.3065

Обучение LinearSVM...
LinearSVM            | macro: 0.4865 | weighted: 0.5699

Обучение SGDClassifier...
SGDClassifier        | macro: 0.4526 | weighted: 0.5346

Обучение kNN...
kNN                  | macro: 0.2729 | weighted: 0.4168
Признаки: BERT weighted mean embeddings

Обучение MultinomialNB...
MultinomialNB        | macro: 0.2079 | weighted: 0.3950

Обучение LinearSVM...
LinearSVM            | macro: 0.5355 | weighted: 0.6171

Обучение SGDClassifier...
SGDClassifier        | macro: 0.4832 | weighted: 0.5591

Обучение kNN.

In [ ]:
import numpy as np
import re
import torch
from datasets import load_dataset
from gensim.models import Word2Vec
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
from transformers import BertTokenizer, BertModel
from tqdm import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split

categories = [
    'comp.sys.ibm.pc.hardware',
    'comp.sys.mac.hardware',
    'comp.graphics',
    'comp.windows.x'
]

data = fetch_20newsgroups(
    subset='all',
    categories=categories,
    shuffle=True,
    random_state=42
)

texts = data.data
labels = data.target

train_texts, test_texts, train_labels, test_labels = train_test_split(
    texts,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

def tokenize_text(text):
    return re.findall(r'\b\w+\b', text.lower())

train_sentences = [tokenize_text(text) for text in train_texts]
test_sentences = [tokenize_text(text) for text in test_texts]

w2v_model = Word2Vec(
    sentences=train_sentences,
    vector_size=100,
    window=5,
    min_count=5,
    workers=4,
    sg=1
)

def get_w2v_mean_embeddings(sentences, model, vector_size):
    embeddings = []
    for sentence in sentences:
        valid_words = [word for word in sentence if word in model.wv.key_to_index]
        if valid_words:
            mean_vec = np.mean(model.wv[valid_words], axis=0)
        else:
            mean_vec = np.zeros(vector_size)
        embeddings.append(mean_vec)
    return np.array(embeddings)

X_train_w2v = get_w2v_mean_embeddings(train_sentences, w2v_model, 100)
X_test_w2v = get_w2v_mean_embeddings(test_sentences, w2v_model, 100)

clf_w2v = Pipeline([
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(max_iter=2000, C=5, class_weight='balanced'))
])

clf_w2v.fit(X_train_w2v, train_labels)

preds_w2v = clf_w2v.predict(X_test_w2v)
f1_w2v_macro = f1_score(test_labels, preds_w2v, average='macro')
f1_w2v_weighted = f1_score(test_labels, preds_w2v, average='weighted')

print(f"F1-score (macro)    для Word2Vec: {f1_w2v_macro:.4f}")
print(f"F1-score (weighted) для Word2Vec: {f1_w2v_weighted:.4f}")

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertModel.from_pretrained('bert-base-uncased')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
bert_model.to(device)
bert_model.eval()

def get_bert_embeddings(texts, batch_size=64):
    cls_embeddings = []
    mean_embeddings = []

    for i in tqdm(range(0, len(texts), batch_size), desc="Обработка текстов BERT"):
        batch_texts = texts[i:i + batch_size]

        inputs = tokenizer(batch_texts, padding=True, truncation=True, max_length=256, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = bert_model(**inputs)

        last_hidden = outputs.last_hidden_state

        batch_cls = last_hidden[:, 0, :].cpu().numpy()
        cls_embeddings.extend(batch_cls)

        mask = inputs['attention_mask'].unsqueeze(-1).expand(last_hidden.size()).float()

        weights = torch.softmax(last_hidden.masked_fill(mask == 0, -1e9), dim=1)

        emb = torch.sum(last_hidden * weights, dim=1)

        batch_weighted = emb.cpu().numpy()
        mean_embeddings.extend(batch_weighted)

    return np.array(cls_embeddings), np.array(mean_embeddings)

X_train_bert_cls, X_train_bert_mean = get_bert_embeddings(train_texts)

X_test_bert_cls, X_test_bert_mean = get_bert_embeddings(test_texts)

clf_bert_cls = Pipeline([
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(max_iter=2000, C=5, class_weight='balanced'))
])
clf_bert_cls.fit(X_train_bert_cls, train_labels)

preds_bert_cls = clf_bert_cls.predict(X_test_bert_cls)
f1_bert_cls_macro = f1_score(test_labels, preds_bert_cls, average='macro')
f1_bert_cls_weighted = f1_score(test_labels, preds_bert_cls, average='weighted')

clf_bert_mean = Pipeline([
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(max_iter=2000, C=5, class_weight='balanced'))
])
clf_bert_mean.fit(X_train_bert_mean, train_labels)

preds_bert_mean = clf_bert_mean.predict(X_test_bert_mean)
f1_bert_mean_macro = f1_score(test_labels, preds_bert_mean, average='macro')
f1_bert_mean_weighted = f1_score(test_labels, preds_bert_mean, average='weighted')

print("\nРезультат")
print(f"Word2Vec (macro):    {f1_w2v_macro:.4f}")
print(f"Word2Vec (weighted): {f1_w2v_weighted:.4f}\n")

print(f"BERT CLS (macro):    {f1_bert_cls_macro:.4f}")
print(f"BERT CLS (weighted): {f1_bert_cls_weighted:.4f}\n")

print(f"BERT mean (macro):   {f1_bert_mean_macro:.4f}")
print(f"BERT mean (weighted):{f1_bert_mean_weighted:.4f}")

F1-score (macro)    для Word2Vec: 0.8171
F1-score (weighted) для Word2Vec: 0.8174


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Обработка текстов BERT: 100%|██████████| 13/13 [00:13<00:00,  1.04s/it]



Результат
Word2Vec (macro):    0.8171
Word2Vec (weighted): 0.8174

BERT CLS (macro):    0.6866
BERT CLS (weighted): 0.6868

BERT mean (macro):   0.7692
BERT mean (weighted):0.7695
